# Notebook 4. Sparse, dense e híbrido



Objetivo: comparar la recuperación léxica (BM25) y la densa (embeddings), entender cuándo gana cada una y
combinarlas. Primero en un corpus de control con verdad de referencia por documento y después en el PDF.

- **V2**: BM25 en español sobre los fragmentos de V1.
- **V3**: híbrido BM25 + denso con RRF.

TF-IDF, la idf de Robertson, la combinación convexa y el mismo híbrido en una base de datos vectorial (Qdrant)
están en `anexos/A2_busqueda_avanzada.ipynb`.

Referencias:
- bm25s: https://huggingface.co/blog/xhluca/bm25s
- Fusión en recuperación híbrida (RRF y combinación convexa): https://arxiv.org/abs/2210.11934

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


## 1. Corpus de control: SQAC

SQAC (PlanTL-GOB-ES/SQAC, licencia CC BY-SA 4.0) son preguntas en español sobre párrafos de Wikipedia,
Wikinews y AnCora. Lo usamos como tarea de recuperación: el documento correcto de cada pregunta es su párrafo.
Con `FAST_MODE` el corpus son los párrafos de dev y test y se toman 200 preguntas.

Limitación: las preguntas de SQAC se escribieron mirando el párrafo y comparten vocabulario con él, lo que en
principio ayuda a BM25. Compruébalo con los resultados de este notebook.

In [2]:
import numpy as np
import pandas as pd
from rag_lab import evaluation, results
from rag_lab.control_corpus import load_sqac

sqac = load_sqac()
ids = list(sqac["docs"])
texts = [sqac["docs"][i] for i in ids]
queries = sqac["consultas"]
print(f"{len(ids)} párrafos, {len(queries)} preguntas")
print("Ejemplo:", queries[0]["texto"], "->", sqac["docs"][queries[0]["doc"]][:200])

1249 párrafos, 200 preguntas
Ejemplo: ¿Cuántas viviendas han sido arrasadas en Nicaragua por el desastre? -> 14 de octubre de 2011 Varios fenómenos meteorológicos azotan a Centroamérica desde hace unos cinco días, generando miles de damnificados y decenas de muertos. Las comisiones respectivas de cada país h


In [3]:
def evaluate_searcher(search, k=10):
    """search(query, k) -> [(doc_id, puntuación)]. Cada documento cuenta como una 'página'."""
    rows = [{"recuperadas": [[d] for d, _ in search(q["texto"], k)], "correctas": [q["doc"]]} for q in queries]
    return evaluation.evaluate_retrieval(rows)

def register_control(version, technique, metrics):
    results.register({"version": version, "notebook": "04", "tecnica": technique, "corpus": "sqac", **metrics})
    return metrics

## 2. BM25 escrito a mano

BM25 es la búsqueda por palabras clave de toda la vida (la de Elasticsearch o Lucene). La idea:

- un documento es más relevante si contiene las palabras de la consulta (**tf**, frecuencia del término),
- sobre todo si son palabras raras en la colección (**idf**: "proyecto" aparece en todas partes, "Dative" no),
- pero repetir una palabra diez veces no lo hace diez veces más relevante (**saturación**),
- y un documento largo contiene más palabras solo por ser largo (**normalización por longitud**).

La fórmula de cada término de la consulta:

`idf(t) · tf · (k1 + 1) / (tf + k1 · (1 − b + b · longitud / longitud_media))`

`k1` controla cuánto cuenta repetir un término (satura antes con k1 pequeño) y `b` cuánto se penaliza
un documento largo (b = 0: nada; b = 1: normalización completa). Usamos stemmer y stopwords en español:
"proyectos" y "proyecto" cuentan como la misma palabra, y "de", "la" o "que" no cuentan.

In [4]:
from bm25s.stopwords import STOPWORDS_SPANISH
import math
import re
from collections import Counter, defaultdict

import Stemmer

stemmer = Stemmer.Stemmer("spanish")
stop = set(STOPWORDS_SPANISH)

def tokenize_es(text):
    return [stemmer.stemWord(w) for w in re.findall(r"\w\w+", text.lower()) if w not in stop]

class ManualBM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        tokens = [tokenize_es(t) for t in texts]
        self.lengths = np.array([len(t) for t in tokens])
        self.avg_len = self.lengths.mean()
        self.postings = defaultdict(list)  # término -> [(documento, tf)]
        for d, toks in enumerate(tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((d, tf))
        n = len(texts)
        self.idf = {t: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5)) for t, p in self.postings.items()}

    def score_docs(self, query):
        scores = np.zeros(len(self.lengths))
        for term in tokenize_es(query):
            for d, tf in self.postings.get(term, []):
                denom = self.k1 * (1 - self.b + self.b * self.lengths[d] / self.avg_len)
                scores[d] += self.idf[term] * tf * (self.k1 + 1) / (tf + denom)
        return scores

    def search(self, query, k=10):
        scores = self.score_docs(query)
        order = np.argsort(-scores)[:k]
        return [(ids[i], float(scores[i])) for i in order if scores[i] > 0]

manual_bm25 = ManualBM25(texts)
register_control("S-bm25-manual", "BM25 a mano (k1=1.5, b=0.75)", evaluate_searcher(manual_bm25.search))

{'recall@5': 0.775, 'recall@10': 0.84, 'MRR@10': 0.6694186507936507}

In [5]:
# Barrido de k1 y b
sweep = []
for k1 in [0.5, 1.5]:
    for b in [0.0, 0.75]:
        m = evaluate_searcher(ManualBM25(texts, k1, b).search)
        sweep.append({"k1": k1, "b": b, "MRR@10": round(m["MRR@10"], 3)})
pd.DataFrame(sweep).pivot(index="k1", columns="b", values="MRR@10")

b,0.00,0.75
k1,,
0.5,0.648,0.669
1.5,0.650,0.669


## 3. bm25s con stemmer y stopwords en español

`bm25s` precalcula las puntuaciones en una matriz dispersa y busca mucho más rápido. Con los mismos k1 y b
debería dar resultados casi iguales a los nuestros (la tokenización no es idéntica).

In [6]:
import time
from rag_lab.indexes import BM25Index

units = [{"id": i, "texto": t, "paginas": [i]} for i, t in zip(ids, texts)]
bm25s_idx = BM25Index(units, k1=1.5, b=0.75)

for name, searcher in [("a mano", manual_bm25.search), ("bm25s", bm25s_idx.search)]:
    t0 = time.perf_counter()
    for q in queries:
        searcher(q["texto"], 10)
    print(f"{name}: {(time.perf_counter() - t0) / len(queries) * 1000:.2f} ms por consulta")
register_control("S-bm25s", "bm25s (stemmer y stopwords en español)", evaluate_searcher(bm25s_idx.search))

a mano: 0.81 ms por consulta
bm25s: 0.31 ms por consulta


{'recall@5': 0.775, 'recall@10': 0.84, 'MRR@10': 0.6694186507936507}

## 4. Búsqueda densa con BGE-M3

Mismo corpus en un índice de Chroma. Los embeddings se calculan una vez y quedan en la caché.

In [7]:
from rag_lab.indexes import DenseIndex

dense = DenseIndex("sqac", units, path=CFG.CACHE_DIR / "chroma_control")
register_control("S-denso", "Denso BGE-M3", evaluate_searcher(dense.search))
results.table("sqac")[["tecnica", "recall@5", "recall@10", "MRR@10"]]

,tecnica,recall@5,recall@10,MRR@10
version,,,,
S-bm25-manual,"BM25 a mano (k1=1.5, b=0.75)",0.775,0.84,0.669419
S-bm25s,bm25s (stemmer y stopwords en español),0.775,0.84,0.669419
S-denso,Denso BGE-M3,0.885,0.91,0.769006


## 5. Cuándo gana cada uno

Buscamos en los resultados reales una pregunta donde BM25 pone el párrafo correcto el primero y el denso no lo
tiene en su top-10, y otra al revés.

In [8]:
def position(ranking, doc):
    ids_ranking = [d for d, _ in ranking]
    return ids_ranking.index(doc) + 1 if doc in ids_ranking else None

rank_bm25 = {q["id"]: bm25s_idx.search(q["texto"], 10) for q in queries}
rank_dense = {q["id"]: dense.search(q["texto"], 10) for q in queries}
cases = {"BM25 acierta, denso falla": [], "denso acierta, BM25 falla": []}
for q in queries:
    pb, pd_ = position(rank_bm25[q["id"]], q["doc"]), position(rank_dense[q["id"]], q["doc"])
    if pb == 1 and pd_ is None:
        cases["BM25 acierta, denso falla"].append(q)
    if pd_ == 1 and pb is None:
        cases["denso acierta, BM25 falla"].append(q)

for title, group in cases.items():
    print(f"=== {title}: {len(group)} preguntas")
    if group:
        q = group[0]
        loser = rank_dense if title.startswith("BM25") else rank_bm25
        print("Pregunta:", q["texto"])
        print("Correcto:", sqac["docs"][q["doc"]][:250])
        first_hit = loser[q["id"]][0][0] if loser[q["id"]] else None
        print("El otro devuelve primero:", sqac["docs"][first_hit][:250] if first_hit else "(nada)", "\n")

=== BM25 acierta, denso falla: 6 preguntas
Pregunta: ¿Quién es Gerry Hunsicker?
Correcto: Por segunda ocasión en los últimos tres años el fantasma de las lesiones en las rodillas de los peloteros volvió a causar mella en el equipo titular de los Astros de Houston. En esta ocasión le correspondió al lanzador estrella, Shane Reynolds, integ
El otro devuelve primero: Diseño y análisis 
En el guion original del episodio Space Seed, escrito por Carey Wilber, el personaje de Khan era un superhombre de origen nórdico llamado Harold Erricsen. El primer borrador del guion presentó al personaje como John Ericssen, uno d 

=== denso acierta, BM25 falla: 10 preguntas
Pregunta: ¿En qué competición de fútbol se ha proclamado campeón el equipo de Uruguay?
Correcto: 24 de julio de 2011 En el estadio Monumental, la selección de Uruguay ganó la Copa América 2011 tras derrotar a su par de Paraguay por 3-0. A los 2' del primer tiempo Diego Forlán metió un centro que cabeceó Lugano y pudo contener el arque

## 6. Híbrido con RRF

**RRF** (*Reciprocal Rank Fusion*) suma `1 / (k + posición)` de cada lista. Solo usa posiciones, así que no hace
falta normalizar puntuaciones de escalas tan distintas como las de BM25 (0 a 20 o más) y el coseno (0 a 1).
Con k = 60, el valor habitual, ser el primero de una lista vale 1/61 y ser el décimo, 1/70.

In [9]:
def rrf_manual(rankings, k=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for position_, (doc, _) in enumerate(ranking, start=1):
            scores[doc] += 1 / (k + position_)
    return sorted(scores.items(), key=lambda x: -x[1])

N = 50  # candidatos de cada lista
cand_bm25 = {q["id"]: bm25s_idx.search(q["texto"], N) for q in queries}
cand_dense = {q["id"]: dense.search(q["texto"], N) for q in queries}
text_to_id = {q["texto"]: q["id"] for q in queries}

def search_rrf(query, k=10):
    qid = text_to_id[query]
    return rrf_manual([cand_dense[qid], cand_bm25[qid]])[:k]

register_control("S-rrf", "Híbrido RRF (k=60)", evaluate_searcher(search_rrf))

{'recall@5': 0.875, 'recall@10': 0.9, 'MRR@10': 0.7563749999999999}

In [10]:
results.table("sqac")[["tecnica", "recall@5", "recall@10", "MRR@10"]]

,tecnica,recall@5,recall@10,MRR@10
version,,,,
S-bm25-manual,"BM25 a mano (k1=1.5, b=0.75)",0.775,0.84,0.669419
S-bm25s,bm25s (stemmer y stopwords en español),0.775,0.84,0.669419
S-denso,Denso BGE-M3,0.885,0.91,0.769006
S-rrf,Híbrido RRF (k=60),0.875,0.90,0.756375


In [11]:
# TODO: prueba RRF con k = 1, 10, 60 y 200 (parámetro k de rrf_manual) y compara el MRR@10.
# ¿Cambia mucho el resultado? ¿Qué posiciones salen ganando con un k pequeño?
for k_rrf in [1, 10, 60, 200]:
    def search_rrf_k(query, k=10):
        # Completa: como search_rrf, pero pasando k=k_rrf a rrf_manual
        return []
    print(f"k = {k_rrf:3d}  MRR@10 = {evaluate_searcher(search_rrf_k)['MRR@10']:.3f}")

k =   1  MRR@10 = 0.000
k =  10  MRR@10 = 0.000
k =  60  MRR@10 = 0.000
k = 200  MRR@10 = 0.000


## 7. En el PDF: V2 y V3

Mismos fragmentos que V1 (por cabeceras). V2 usa solo BM25; V3 fusiona BM25 y denso con RRF.

In [12]:
from rag_lab import models, versions

questions = evaluation.load_questions()
for version in ["V2", "V3"]:
    evaluation.evaluate_rag(versions.build(version), questions)
results.table("pdf").loc[["V1", "V2", "V3"], ["tecnica", "recall@5", "recall@10", "MRR@10",
                                                 "recall@5_texto", "recall@5_tabla", "recall@5_visual"]]

,tecnica,recall@5,recall@10,MRR@10,recall@5_texto,recall@5_tabla,recall@5_visual
version,,,,,,,
V1,Chunking por cabeceras y tablas completas + de...,1.0,1.0,0.808333,1.0,1.0,1.000000
V2,"BM25 (bm25s, stemmer y stopwords en español)",0.9,0.9,0.760000,1.0,1.0,0.666667
V3,Híbrido BM25 + denso con RRF,1.0,1.0,0.858333,1.0,1.0,1.000000


Fíjate en las columnas por tipo de pregunta: ¿en qué tipo gana BM25 (V2) a la búsqueda densa (V1) y en cuál
pierde? ¿Qué hace el híbrido (V3) con cada uno?

In [13]:
models.release()

## Qué hemos visto

- BM25 acierta cuando la pregunta repite palabras exactas del documento; el denso, cuando las dice con otras palabras.
- k1 y b ajustan la saturación de la frecuencia y la penalización por longitud.
- RRF fusiona listas usando solo posiciones: no hace falta normalizar puntuaciones ni elegir pesos.

## Ejercicio propuesto

Repite la comparación de BM25, denso y RRF sobre SQAC con Qwen3-Embedding-0.6B como modelo denso y
comprueba si cambia cuál es el mejor método.